# Apple App Store — Market Analysis

**Author:** Edy Santos  
**Tools:** PostgreSQL · Python · Plotly  
**Dataset:** [Kaggle — App Store Apps (~7,200 apps)](https://www.kaggle.com/datasets/ramamet4/app-store-apple-data-set-10k-apps)

---

## Objective

In this scenario, the stakeholder is an app developer who wishes to maximise their efforts towards a marketable, successful app. To achieve that, this professional seeks to find out what the most popular app categories are, what price to set, and how to maximise platform user ratings.

**Questions explored:**
- Which genres dominate the store — and which are underserved?
- Do paid apps outperform free apps in user ratings?
- What app characteristics (description length, language support, price) correlate with higher ratings?
- Within low-rated genres, which standout apps beat the category average?

---

## Structure

| Section | Focus |
|---|---|
| 1 | Setup & Database Connection |
| 2 | Data Quality Checks |
| 3 | Genre Distribution |
| 4 | Ratings Landscape |
| 5 | Low-Rated Genres — Standout & Worst Apps |
| 6 | Description Length vs Ratings |
| 7 | Language Support vs Ratings |
| 8 | Paid vs Free |
| 9 | Price Range Distribution |
| 10 | Visualisations |
| 11 | Recommendations |

---
## Section 1 — Setup & Database Connection

We connect to a local PostgreSQL instance using `psycopg2` and wrap queries in a helper function that returns results as a `pandas` DataFrame for easy display and downstream use.

In [ ]:
import psycopg2
import pandas as pd
import plotly.express as px
from IPython.display import display

# Update these credentials to match your local PostgreSQL setup
conn = psycopg2.connect(
    dbname='postgres_anydb',
    user='postgres_anyuser',
    password='private_passcode_here',
    host='localhost',
    port=XXXX
)

def run_query(sql):
    """Execute a SQL query and return results as a DataFrame."""
    return pd.read_sql_query(sql, conn)

print('Connection established.')

---
## Section 2 — Data Quality Checks

Before any analysis, we need to check if both datasets have the exact same app list, as well as identify any null or missing values. This step is essential — any discrepancies found here would affect how we interpret everything downstream.

Let's look deep into it.

In [ ]:
# Check unique app counts in each table — should match
count_dataset = run_query("""
    SELECT COUNT(DISTINCT id) AS unique_app_ids_dataset
    FROM applestore_dataset;
""")

count_description = run_query("""
    SELECT COUNT(DISTINCT id) AS unique_app_ids_description
    FROM applestoredescription;
""")

print('Unique app IDs — Dataset table:')
display(count_dataset)
print('Unique app IDs — Description table:')
display(count_description)

In [ ]:
# Identify apps present in one table but missing from the other
mismatch = run_query("""
    SELECT 'applestore_dataset' AS source, track_name
    FROM applestore_dataset
    WHERE track_name NOT IN (SELECT track_name FROM applestoredescription)
    UNION ALL
    SELECT 'applestoredescription' AS source, track_name
    FROM applestoredescription
    WHERE track_name NOT IN (SELECT track_name FROM applestore_dataset);
""")

print(f'Mismatched records: {len(mismatch)}')
display(mismatch.head(10))

In [ ]:
# Check for missing values in critical fields
missing_dataset = run_query("""
    SELECT COUNT(*) AS missing_values_dataset
    FROM applestore_dataset
    WHERE track_name IS NULL
       OR user_rating IS NULL
       OR prime_genre IS NULL;
""")

missing_description = run_query("""
    SELECT COUNT(*) AS missing_values_description
    FROM applestoredescription
    WHERE app_desc IS NULL;
""")

print('Missing values — Dataset:')
display(missing_dataset)
print('Missing values — Description:')
display(missing_description)

Since we did not find any discrepancies between the two tables, we can now proceed to the next step.

---
## Section 3 — Genre Distribution

So what are the **most numerous genres**?

High app counts signal saturated markets. An outlier in one genre could suggest either a dominant category or an underserved opportunity depending on the quality of existing apps.

In [ ]:
genre_dist = run_query("""
    SELECT prime_genre,
           COUNT(*) AS num_apps
    FROM applestore_dataset
    GROUP BY prime_genre
    ORDER BY num_apps DESC;
""")

display(genre_dist)

# Bar chart — top 10 genres by app count
top10 = genre_dist.head(10)
fig = px.bar(
    top10,
    x='prime_genre',
    y='num_apps',
    title='Top 10 Genres by Number of Apps',
    labels={'prime_genre': 'Genre', 'num_apps': 'Number of Apps'},
    color='num_apps',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-45, plot_bgcolor='white')
fig.show()

# Stacked bar chart — genres and user rating distribution (top 10 genres)
# Fetches count of apps per genre per rating value for the top 10 genres by volume
stacked_data = run_query("""
    WITH top_genres AS (
        SELECT prime_genre
        FROM applestore_dataset
        GROUP BY prime_genre
        ORDER BY COUNT(*) DESC
        LIMIT 10
    )
    SELECT a.prime_genre,
           a.user_rating::text AS user_rating,
           COUNT(*) AS app_count
    FROM applestore_dataset a
    JOIN top_genres t ON a.prime_genre = t.prime_genre
    GROUP BY a.prime_genre, a.user_rating
    ORDER BY a.prime_genre, a.user_rating;
""")

fig_stacked = px.bar(
    stacked_data,
    x='prime_genre',
    y='app_count',
    color='user_rating',
    title='Genres & Stacked User Ratings Total',
    labels={'prime_genre': 'Genre', 'app_count': 'Apps Number', 'user_rating': 'User Rating'},
    color_discrete_sequence=px.colors.sequential.Blues
)
fig_stacked.update_layout(xaxis_tickangle=-45, plot_bgcolor='white', barmode='stack')
fig_stacked.show()

An outlier has been identified: the majority of store apps fall under the **Games** category, with 3,862 apps. This suggests that the Games category might be more competitive and therefore harder to succeed in.

---
## Section 4 — Ratings Landscape

Overall, what is the **average user rating**?

We start with overall rating stats, then examine which genres are consistently underperforming. A `WITH RECURSIVE` query generates every possible rating value (0.0, 0.5 … 5.0) so that genre-rating combinations with zero apps still appear in the output — important for an accurate distribution.

In [ ]:
rating_stats = run_query("""
    SELECT MIN(user_rating)                       AS min_rating,
           MAX(user_rating)                       AS max_rating,
           ROUND(AVG(user_rating)::numeric, 2)    AS avg_rating
    FROM applestore_dataset;
""")

display(rating_stats)

From this, we can conclude that an app with a **3.5 rating** would be around the average for the Apple App Store.

In [ ]:
# Rating distribution across top 10 genres
rating_dist = run_query("""
    WITH RECURSIVE ratings AS (
        SELECT 0.0::numeric AS rating
        UNION ALL
        SELECT rating + 0.5 FROM ratings WHERE rating < 5.0
    ),
    genre_totals AS (
        SELECT a.prime_genre,
               SUM(COUNT(a.user_rating)) OVER (PARTITION BY a.prime_genre) AS total_ratings
        FROM ratings r
        LEFT JOIN applestore_dataset a ON r.rating = a.user_rating
        GROUP BY a.prime_genre, r.rating
    ),
    top_10_genres AS (
        SELECT DISTINCT prime_genre
        FROM genre_totals
        ORDER BY total_ratings DESC
        LIMIT 10
    )
    SELECT gt.prime_genre,
           r.rating   AS user_rating,
           COUNT(a.user_rating) AS genre_count
    FROM ratings r
    LEFT JOIN applestore_dataset a ON r.rating = a.user_rating
    JOIN top_10_genres gt ON a.prime_genre = gt.prime_genre
    GROUP BY gt.prime_genre, r.rating
    ORDER BY gt.prime_genre, r.rating;
""")

display(rating_dist.head(20))

In [ ]:
# 10 lowest-rated genres
low_rated = run_query("""
    SELECT prime_genre,
           ROUND(AVG(user_rating)::numeric, 2) AS avg_rating
    FROM applestore_dataset
    GROUP BY prime_genre
    ORDER BY avg_rating ASC
    LIMIT 10;
""")

display(low_rated)

# Bar chart
fig2 = px.bar(
    low_rated,
    x='prime_genre',
    y='avg_rating',
    title='10 Lowest-Rated Genres — Average User Rating',
    labels={'prime_genre': 'Genre', 'avg_rating': 'Average Rating'},
    color='avg_rating',
    color_continuous_scale='Reds_r'
)
fig2.update_layout(xaxis_tickangle=-45, plot_bgcolor='white', yaxis=dict(range=[0, 4]))
fig2.show()

This looks promising — there are **ten categories (genres) rated at 3.25 or below** on average. Examining these could reveal opportunities in areas that are currently underserved.

---
## Section 5 — Low-Rated Genres: Standout & Worst Apps

Next, let's identify which apps have **higher ratings within these lower-rated genres**. To focus on a more relevant subset, we'll limit our search to apps that have been rated at least **5,000 times**.

We use a two-step CTE: first identify the low-rated genres, then rank apps within each genre by rating and volume using `RANK() OVER (PARTITION BY ...)`.

In [ ]:
# Best-rated app (5,000+ ratings) in each of the 10 lowest-rated genres
standout_apps = run_query("""
    WITH low_rated_genres AS (
        SELECT prime_genre,
               ROUND(AVG(user_rating)::numeric, 2) AS avg_rating
        FROM applestore_dataset
        GROUP BY prime_genre
        ORDER BY avg_rating ASC
        LIMIT 10
    ),
    ranked_tracks AS (
        SELECT a.prime_genre,
               a.track_name,
               a.user_rating,
               a.rating_count_tot,
               RANK() OVER (
                   PARTITION BY a.prime_genre
                   ORDER BY a.user_rating DESC, a.rating_count_tot DESC
               ) AS rank
        FROM applestore_dataset AS a
        JOIN low_rated_genres AS lrg ON a.prime_genre = lrg.prime_genre
        WHERE a.rating_count_tot > 5000
    )
    SELECT prime_genre, track_name, user_rating
    FROM ranked_tracks
    WHERE rank = 1
    ORDER BY user_rating DESC;
""")

display(standout_apps)

In [ ]:
# Worst-rated apps in the same low-rated genres (5,000+ ratings)
worst_apps = run_query("""
    WITH low_rated_genres AS (
        SELECT prime_genre,
               ROUND(AVG(user_rating)::numeric, 2) AS avg_rating_genre,
               COUNT(*) AS app_count
        FROM applestore_dataset
        GROUP BY prime_genre
        ORDER BY avg_rating_genre ASC
        LIMIT 10
    ),
    ranked_tracks AS (
        SELECT a.prime_genre,
               a.track_name,
               a.user_rating,
               a.price,
               RANK() OVER (
                   PARTITION BY a.prime_genre
                   ORDER BY a.user_rating ASC, a.rating_count_tot DESC
               ) AS rank
        FROM applestore_dataset AS a
        JOIN low_rated_genres AS lrg ON a.prime_genre = lrg.prime_genre
        WHERE a.rating_count_tot > 5000
    )
    SELECT lrg.prime_genre, rt.track_name,
           rt.user_rating AS lowrated_app_user_rating,
           rt.price,
           lrg.app_count,
           lrg.avg_rating_genre
    FROM ranked_tracks AS rt
    JOIN low_rated_genres AS lrg ON rt.prime_genre = lrg.prime_genre
    WHERE rt.rank = 1
    ORDER BY rt.user_rating ASC;
""")

display(worst_apps)

From the table above, we can observe the apps with the ten lowest ratings, along with their respective prices and the total number of apps in these low-rated categories. Also important to note: **almost all of them are free**, which raises an important question — what about the paid ones?

---
## Section 6 — Description Length vs Ratings

On a side note, some apps have longer descriptions than others. **Could the length of the description influence the ratings?**

Let's define short descriptions as those with fewer than 500 characters, medium descriptions as those between 500 and 1,000 characters, and long descriptions as those exceeding 1,000 characters.

In [ ]:
desc_length = run_query("""
    SELECT CASE
               WHEN LENGTH(b.app_desc) < 500          THEN 'Short'
               WHEN LENGTH(b.app_desc) BETWEEN 500
                    AND 1000                           THEN 'Medium'
               ELSE                                        'Long'
           END AS description_length_bucket,
           ROUND(AVG(a.user_rating)::numeric, 2) AS average_rating
    FROM applestore_dataset AS a
    JOIN applestoredescription AS b ON a.id = b.id
    GROUP BY description_length_bucket
    ORDER BY average_rating DESC;
""")

display(desc_length)

It appears that users tend to prefer apps with **detailed descriptions** that explain the app's purpose, features, and other key details. Therefore, apps with descriptions under 500 characters seem less favoured.

---
## Section 7 — Language Support vs Ratings

Next, we could explore whether **highly rated apps support a significant number of languages**.

While some apps are only available in a single language — often English — others offer a broad selection, with some supporting over 30 languages. It would be interesting to examine whether the number of supported languages has an impact on app ratings.

In [ ]:
lang_support = run_query("""
    SELECT CASE
               WHEN lang_num < 10                THEN '< 10 Languages'
               WHEN lang_num BETWEEN 10 AND 30   THEN '10 - 30 Languages'
               ELSE                                   '> 30 Languages'
           END AS language_bucket,
           ROUND(AVG(user_rating)::numeric, 2) AS avg_rating
    FROM applestore_dataset
    GROUP BY language_bucket
    ORDER BY avg_rating DESC;
""")

display(lang_support)

That's an interesting finding: **something between 10–30 languages could do very well**. Supporting more languages than necessary showed diminishing returns on ratings.

---
## Section 8 — Paid vs Free

First, what about **free apps**? Are they better rated than paid apps? Which kind are more numerous?

It's interesting to note that in the market some apps are free to download and sign up, but likely require payments for extra features — a strategy known as **Freemium**. Tinder is a good example of this.

In [ ]:
paid_vs_free = run_query("""
    SELECT CASE
               WHEN price = 0 THEN 'Free'
               ELSE                'Paid'
           END AS app_type,
           COUNT(*) AS number_of_apps,
           ROUND(AVG(user_rating)::numeric, 2) AS avg_rating
    FROM applestore_dataset
    GROUP BY app_type
    ORDER BY avg_rating DESC;
""")

display(paid_vs_free)

# Bar chart — paid vs free
fig3 = px.bar(
    paid_vs_free,
    x='app_type',
    y='avg_rating',
    color='app_type',
    text='number_of_apps',
    title='Average Rating: Paid vs Free Apps',
    labels={'app_type': 'App Type', 'avg_rating': 'Average Rating'},
    color_discrete_map={'Paid': 'steelblue', 'Free': 'salmon'}
)
fig3.update_traces(texttemplate='%{text} apps', textposition='outside')
fig3.update_layout(plot_bgcolor='white', yaxis=dict(range=[0, 4.5]), showlegend=False)
fig3.show()

We can observe that the average ratings of both categories are **roughly the same**, which certainly affects our analysis. As for the number of apps in each category, while free apps are more numerous, we do not have the data to determine whether they are freemium-based.

---
## Section 9 — Price Range Distribution

Since we are already analysing whether apps are free or paid, as well as their distribution, **let's also examine the price range**.

In [ ]:
price_dist = run_query("""
    SELECT CASE
               WHEN price = 0                  THEN 'Free'
               WHEN price BETWEEN 0.01 AND 1   THEN '$0.01 - $1.00'
               ELSE CONCAT(
                        '$',
                        (FLOOR((price - 1) / 2) * 2 + 1)::numeric(10,2)::text,
                        ' - $',
                        (FLOOR((price - 1) / 2) * 2 + 2.99)::numeric(10,2)::text
                    )
           END AS price_range,
           COUNT(*) AS num_apps
    FROM applestore_dataset
    GROUP BY price_range
    ORDER BY MIN(price);
""")

display(price_dist)

The table reveals a trend where **most paid apps are priced between $1 and $3**. At this stage, we can say that there is plenty of room for bringing a new app to market in News, Entertainment, and Navigation.

---
## Section 10 — Visualisations

The bubble chart below highlights the **10 lowest-rated genres**, each with over 5,000 user ratings, along with their respective quantities. Built directly from the SQL output — an end-to-end workflow: PostgreSQL → DataFrame → Plotly.

- **X axis:** Genre
- **Y axis:** Average genre rating
- **Bubble size:** Number of apps in that genre
- **Hover:** Genre name, worst-rated app, and app count

In [ ]:
low_rated_viz = run_query("""
    WITH low_rated_genres AS (
        SELECT prime_genre,
               ROUND(AVG(user_rating)::numeric, 2) AS avg_rating_genre,
               COUNT(*) AS app_count
        FROM applestore_dataset
        GROUP BY prime_genre
        ORDER BY avg_rating_genre ASC
        LIMIT 10
    ),
    ranked_tracks AS (
        SELECT a.prime_genre,
               a.track_name,
               a.user_rating,
               RANK() OVER (
                   PARTITION BY a.prime_genre
                   ORDER BY a.user_rating ASC, a.rating_count_tot DESC
               ) AS rank
        FROM applestore_dataset AS a
        JOIN low_rated_genres AS lrg ON a.prime_genre = lrg.prime_genre
        WHERE a.rating_count_tot > 5000
    )
    SELECT lrg.prime_genre, rt.track_name,
           rt.user_rating AS low_rated_app_user_rating,
           lrg.app_count,
           lrg.avg_rating_genre
    FROM ranked_tracks AS rt
    JOIN low_rated_genres AS lrg ON rt.prime_genre = lrg.prime_genre
    WHERE rt.rank = 1
    ORDER BY lrg.avg_rating_genre ASC;
""")

fig4 = px.scatter(
    low_rated_viz,
    x='prime_genre',
    y='avg_rating_genre',
    size='app_count',
    hover_name='prime_genre',
    hover_data={'track_name': True, 'app_count': True, 'low_rated_app_user_rating': True},
    title='Average Genre Ratings vs Low-rated Genres (Over 5,000 entries)',
    labels={
        'prime_genre':              'Low-rated Genres (Over 5000 entries)',
        'avg_rating_genre':         'Average Genre Ratings',
        'app_count':                'Number of Apps',
        'track_name':               'Worst-rated App',
        'low_rated_app_user_rating':'App Rating'
    },
    color='avg_rating_genre',
    color_continuous_scale='RdYlGn'
)
fig4.update_layout(
    xaxis_tickangle=-45,
    xaxis_tickfont=dict(size=10),
    plot_bgcolor='white',
    yaxis=dict(range=[0, 4])
)
fig4.show()

---
## Section 11 — Recommendations

In conclusion, based on the analysis, we recommend the following:

### 🎯 Target Genre
Develop an app in **underrepresented, low-rated categories** like Catalogs, Finance, and Navigation. These categories offer potential due to their smaller number of apps, which presents less competition. A well-designed app with a superior user experience could address existing gaps and capitalise on the lack of strong contenders in these spaces.

For the **Books** category, while there is uncertainty around whether low ratings apply to the content or e-reader functionality, further investigation could help clarify if this area presents a viable opportunity.

### 💰 Pricing Strategy
Launch a **paid app priced between USD 1.00 and USD 4.00**, or adopt a **Freemium model** — offering basic features for free while charging for advanced options. This model allows you to attract a larger user base initially while still monetising effectively.

### 🌍 Language Support
Target **10 to 30 languages**. Supporting more languages than necessary showed diminishing returns — the sweet spot is broad but targeted localisation.

### 📝 App Description
Invest in a **long, detailed app description**. The data clearly shows that apps with descriptions over 1,000 characters consistently achieve higher average ratings — users respond positively to apps that clearly explain their purpose and features.

---

**Author:** Edy dos Santos  
[GitHub Portfolio](https://github.com/edy-dos-santos/myportfolio)

In [ ]:
# Close the database connection
conn.close()
print('Connection closed.')